In [12]:
import requests
import pandas as pd
import ast

from keys import TMDB_API_KEY

BASE_URL = "https://api.themoviedb.org/3"

# Load genre lookup without hardcoding
df_genres = pd.read_csv("genres.csv") # Listed id, genre name
genre_lookup = dict(zip(df_genres["id"], df_genres["name"])) # Combine ID and word into a list


In [13]:
def search_movie(term):
    url = f"{BASE_URL}/search/movie"
    params = {
        "api_key": TMDB_API_KEY,
        "query": term
    }
    response = requests.get(url, params=params)

    # Check to make sure results are actually being sent
    if response.status_code != 200:
        print("Error: ", response.status_code)
        print(response.text)
        return []


    data = response.json()

    # If something is returned that isn't expected
    if "results" not in data or data["results"] is None:
        print("TMDB returned no results.")
        print(data)
        return []
    
    results = data.get("results", [])

    movies = []
    for m in results:
        movies.append({
            "id": m["id"],
            "title": m["title"],
            "year": (m.get("release_date") or "")[:4],
            "overview": m.get("overview", ""),
            "genre_ids": m.get("genre_ids", [])
        })



    return movies

In [14]:
# Testing Movie lookup without hardcoding
term = input("Search for a movie: ")
movies = search_movie(term)
print(movies[:3])

[{'id': 157336, 'title': 'Interstellar', 'year': '2014', 'overview': 'The adventures of a group of explorers who make use of a newly discovered wormhole to surpass the limitations on human space travel and conquer the vast distances involved in an interstellar voyage.', 'genre_ids': [12, 18, 878]}, {'id': 529107, 'title': "Inside 'Interstellar'", 'year': '2015', 'overview': 'Cast and crew of Christopher Nolan\'s "Interstellar" discuss project origins, the film\'s imagery, ambitions, incorporating IMAX footage, the human element within the film, arm shooting locations outside of Calgary, the set construction and design, working with real corn, mechanical characters, including backstory, design, the blend of practical and digital effects in bringing them to life, the differences in the characters, the human performances behind the characters, the creative process behind the film\'s music, Icelandic locations, vehicle interiors, the processes of simulating the absence of gravity, the cruc

In [37]:
def submit_review(movie_id, rating, genre_ids=None):
    # Validate User review
    if rating < 1 or rating > 5:
        print("You must pick between 1 and 5 stars.")
        return
    
    # Load existing reviews (if any)
    try:
        df = pd.read_csv("reviews.csv")
    except FileNotFoundError:
        df = pd.DataFrame(columns=["movie_id", "rating", "genre_ids"])

    # Add a new row for the new review
    new_row = {
        "movie_id": movie_id,
        "rating": rating,
        "genre_ids": str(genre_ids) if genre_ids is not None else "" # Add empty string if no genre id string to keep csv clean
    }
    # Add new review to the bottom of reviews.csv
    df = pd.concat([df, pd.DataFrame([new_row])], ignore_index=True)

    # Save new review to reviews.csv
    df.to_csv("reviews.csv", index=False)
    print("Your review was saved!")

In [ ]:
# Test submit_review 
first = movies[0]
submit_review(first["id"], 5, first["genre_ids"])


Your review was saved!


In [ ]:
def get_movie_details(movie_id):

    # Get movie details from TMDB
    url = f"{BASE_URL}/movie/{movie_id}"
    params = {
        "api_key": TMDB_API_KEY
    }
    response = requests.get(url, params=params)

    # Check to make sure request is fufilled by TMDB API
    if response.status_code != 200:
        print("TMDB request error:", response.status_code)
        return {}
    
    # Return details in JSON format
    return response.json()

In [ ]:
# Testing get_movie_details
get_movie_details(7512)

{'adult': False,
 'backdrop_path': '/9Z4msMhKvNp63sOB1ZvxPMPvqz3.jpg',
 'belongs_to_collection': None,
 'budget': 30000000,
 'genres': [{'id': 35, 'name': 'Comedy'},
  {'id': 878, 'name': 'Science Fiction'},
  {'id': 12, 'name': 'Adventure'}],
 'homepage': 'https://ahighlight.com',
 'id': 7512,
 'imdb_id': 'tt0387808',
 'origin_country': ['US'],
 'original_language': 'en',
 'original_title': 'Idiocracy',
 'overview': "To test its top-secret Human Hibernation Project, the Pentagon picks the most average European-Americans it can find - an Army private and a prostitute - and sends them to the year 2505 after a series of freak events. But when they arrive, they find a civilization so dumbed-down that they're the smartest people around.",
 'popularity': 9.7198,
 'poster_path': '/6cTHBq49ApwsJaRr3ojlY1cmiXk.jpg',
 'production_companies': [{'id': 25,
   'logo_path': '/qZCc1lty5FzX30aOCVRBLzaVmcp.png',
   'name': '20th Century Fox',
   'origin_country': 'US'},
  {'id': 19985,
   'logo_path': 

In [1]:
def generate_movieboard(top_n=10):
    # Read reviews.csv and check to see if there is reviews
    try:
        df = pd.read_csv("reviews.csv")
    except FileNotFoundError:
        print("No reviews yet.")
        return
    
    # No reviews present?
    if df.empty:
        print("No reviews yet.")
        return
    
    # Convert genre_ids back to lists
    df["genre_ids"] = df["genre_ids"].apply(lambda x: ast.literal_eval(x) if x else []) # Add space if no genre id is present
    
    # group movies by ID
    grouped_ID = df.groupby("movie_id")["rating"].agg(["mean", "count"]).reset_index()
    grouped_ID = grouped_ID.sort_values(by=["mean", "count"], ascending=[False, False])

    # Show top movies based on set value (n)
    print("Top movies:")
    for i, row in grouped_ID.head(top_n).iterrows():
        movie_id = int(row["movie_id"])
        avg = row["mean"]
        count = int(row["count"])

        # Get movie title from TMDB 
        details = get_movie_details(movie_id)
        title = details.get("title", "Unknown title")

        # Convert TMDB Genre IDs to words
        tmdb_genres = details.get("genres", [])
        genre_names = [genre_lookup[g["id"]] for g in tmdb_genres if g["id"] in genre_lookup]

        # Print movieboard entry
        print(f"{title} ({', '.join(genre_names)}) -- Avg: {avg: .2f} from {count} reviews")
    

In [42]:
# Test generate_movieboard
generate_movieboard()

Top movies:
Idiocracy (ID: 7512) -- Avg:  5.00 from 1 reviews
